# Post #5 — Commuter Arbitrage
**CT Town Personas · LODES 2021 + ACS 2022 · CT Science Center engagement**

Identifies high-probability visitor source markets for CT attractions by combining:
- LODES8 origin-destination commute flows (Census LEHD, 2021)
- ACS median household income (5-year estimates, 2022)
- Drive-time band assignment (haversine × piecewise speed, calibrated May 2026)
- Gravity-inspired scoring with distance decay

**Framing note:** Commute flow predicts *geographic familiarity*, which is a hypothesized
leading indicator of leisure visitation — not a direct predictor. These towns have the
lowest geographic friction to visiting, making them the highest-probability marketing
audience. Whether they convert depends on programming, awareness, and fit.

In [ ]:
import subprocess, sys
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'matplotlib', 'pyarrow'],
    stderr=subprocess.DEVNULL,
)
print('Dependencies ready')

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')

# Resolve project root whether Jupyter was launched from posts/ or the project root
_cwd = Path().resolve()
ROOT = _cwd if (_cwd / 'data' / 'processed').exists() else _cwd.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pipeline.drive_time import add_drive_columns
from pipeline.behavior_overlay import apply_behavior_overlay
from pipeline.attractions import weighted_flow, ATTRACTIONS

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})

PROCESSED = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'outputs' / 'ct_science_center'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT)
print('PROCESSED:', PROCESSED)

## 1. Data Loading

In [ ]:
# LODES anchor flows: 169 towns × 4 anchors (Hartford, New Haven, Stamford, Bridgeport)
lodes = pd.read_parquet(PROCESSED / 'lodes_anchor_flows_2021.parquet')

# Features 2022 — 4 rows per town from outer merge across datasets; deduplicate
# by taking the last row, which carries the ACS income data.
features = (
    pd.read_parquet(PROCESSED / 'town_features_2022.parquet')
    .groupby('town', as_index=False)
    .last()
)

# Cluster assignments 2022
clusters = (
    pd.read_parquet(PROCESSED / 'town_clusters.parquet')
    .query('year == 2022')
    .groupby('town', as_index=False)
    .last()[['town', 'cluster_id', 'archetype_label']]
)

print(f'LODES: {lodes.shape[0]} towns × {lodes.shape[1]} cols')
print(f'Features (deduped): {features.shape[0]} towns × {features.shape[1]} cols')
print(f'Clusters: {clusters.shape[0]} towns')
lodes.head(3)

In [ ]:
ANCHORS = ['hartford', 'new_haven', 'stamford', 'bridgeport']
ANCHOR_TOWNS = ['Hartford', 'New Haven', 'Stamford', 'Bridgeport']

df = (
    features
    .merge(lodes.drop(columns='lodes_year'), on='town', how='left', suffixes=('', '_lodes'))
    .merge(clusters, on='town', how='left')
)

# Resolve duplicate flow columns (features already has them from pipeline; prefer LODES)
for a in ANCHORS:
    col, col_l = f'inbound_to_{a}', f'inbound_to_{a}_lodes'
    if col_l in df.columns:
        df[col] = df[col].fillna(df[col_l])
        df.drop(columns=col_l, inplace=True)

df['total_inbound'] = df[[f'inbound_to_{a}' for a in ANCHORS]].sum(axis=1)

# Exclude anchor cities from arbitrage analysis (self-commute inflates their score)
df_ex = df[~df['town'].isin(ANCHOR_TOWNS)].copy()

print(f'{len(df)} total towns, {len(df_ex)} excluding anchor cities')
df[['town', 'inbound_to_hartford', 'inbound_to_new_haven', 'inbound_to_stamford',
    'inbound_to_bridgeport', 'total_inbound']].nlargest(8, 'total_inbound')

## 2. LODES Overview — Top Commuter Towns by Anchor

Context chart: raw inbound flows before applying any attraction-specific weighting.
Shows why a flat total-inbound ranking would be misleading for a Hartford attraction —
Hamden (7,700 New Haven commuters) would outrank West Hartford (6,400 Hartford commuters).

In [ ]:
anchor_labels = {'hartford': 'Hartford', 'new_haven': 'New Haven',
                 'stamford': 'Stamford', 'bridgeport': 'Bridgeport'}
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, anchor, color in zip(axes.flat, ANCHORS, colors):
    col = f'inbound_to_{anchor}'
    top = df_ex.nlargest(12, col)[['town', col]].reset_index(drop=True)
    ax.barh(top['town'][::-1], top[col][::-1], color=color, alpha=0.8)
    ax.set_xlabel('Inbound Workers (LODES 2021)')
    ax.set_title(f'→ {anchor_labels[anchor]}')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('CT LODES 2021 — Top Commuter Towns by Anchor City', y=1.01, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED / 'post05_top_commuter_towns.png', bbox_inches='tight')
plt.show()

## 3. CT Science Center — Attraction-Specific Weighted Flow

Rather than summing all four anchor flows equally, we weight them by relevance to the
specific attraction. For the CT Science Center (Hartford):

| Anchor | Weight | Rationale |
|--------|--------|----------|
| Hartford | 1.0 | Primary — attraction is in Hartford |
| New Haven | 0.3 | Secondary — I-91 corridor, some shared catchment |
| Stamford | 0.0 | Irrelevant — Fairfield County corridor |
| Bridgeport | 0.0 | Irrelevant — same reason |

**These weights are subjective defaults**, documented here as methodology inputs.
A client engagement can recalibrate them using observed ticket-purchase geography.
Under this weighting, Fairfield County towns correctly drop out of the top rankings.

In [ ]:
ATTRACTION_KEY = 'ct_science_center'
ANCHOR_LAT = ATTRACTIONS[ATTRACTION_KEY]['lat']
ANCHOR_LON = ATTRACTIONS[ATTRACTION_KEY]['lon']

df_ex = df_ex.copy()
df_ex['weighted_flow'] = weighted_flow(df_ex, ATTRACTION_KEY)

print('Top 10 by weighted flow (Science Center weighting):')
df_ex[['town', 'weighted_flow', 'inbound_to_hartford', 'inbound_to_new_haven']].nlargest(10, 'weighted_flow')

## 4. Gravity-Inspired Arbitrage Score

Replaces the simple 50/50 percentile average with a gravity-inspired formula that
incorporates distance decay — consistent with 90+ years of retail/tourism site-selection
literature where visit probability decays with distance.

```
score = (flow_pct^α × income_pct^β) / drive_time_min^γ
```

Normalized to 0–1 for interpretability. Default parameters (α=1.0, β=1.0, γ=1.5) are
stored in `WEIGHTS` — see sensitivity analysis in Section 6 for how rankings shift
under alternative parameterizations.

In [ ]:
WEIGHTS = {'alpha': 1.0, 'beta': 1.0, 'gamma': 1.5}

# Add drive time from Science Center
arb = add_drive_columns(df_ex.copy(), ANCHOR_LAT, ANCHOR_LON)
arb = apply_behavior_overlay(arb)

# Percentile ranks (0–1)
flow_pct   = arb['weighted_flow'].rank(pct=True)
income_pct = arb['median_household_income'].rank(pct=True).fillna(0)
drive_safe = arb['drive_time_min'].clip(lower=1)  # prevent division by zero

alpha, beta, gamma = WEIGHTS['alpha'], WEIGHTS['beta'], WEIGHTS['gamma']
raw_score = (flow_pct ** alpha) * (income_pct ** beta) / (drive_safe ** gamma)
arb['arbitrage_score'] = ((raw_score - raw_score.min()) / (raw_score.max() - raw_score.min())).round(4)

print(f'Score distribution: min={arb["arbitrage_score"].min():.3f}, '
      f'median={arb["arbitrage_score"].median():.3f}, max={arb["arbitrage_score"].max():.3f}')

# Drive band summary
print()
print('Drive-time band distribution (all 165 non-anchor towns):')
print(arb['drive_band'].value_counts().to_string())

## 5. Top Source Markets — CT Science Center

In [ ]:
top15 = arb.nlargest(15, 'arbitrage_score')[[
    'town', 'arbitrage_score', 'drive_time_min', 'drive_band',
    'tourism_behavior', 'weighted_flow', 'median_household_income', 'archetype_label'
]].copy().reset_index(drop=True)
top15.index += 1
top15.index.name = 'Rank'

top15_display = top15.copy()
top15_display['median_household_income'] = top15_display['median_household_income'].map(
    lambda x: f'${int(x):,}' if pd.notna(x) else 'n/a'
)
top15_display['weighted_flow'] = top15_display['weighted_flow'].map(lambda x: f'{int(x):,}')
top15_display

In [ ]:
# Scatter: weighted flow vs income, colored by arbitrage score
plot_df = arb.dropna(subset=['median_household_income'])

fig, ax = plt.subplots(figsize=(11, 7))
sc = ax.scatter(
    plot_df['weighted_flow'], plot_df['median_household_income'],
    c=plot_df['arbitrage_score'], cmap='YlOrRd', s=60, alpha=0.75, edgecolors='none',
)
plt.colorbar(sc, ax=ax, label='Arbitrage Score (0–1)')

for _, row in arb.dropna(subset=['median_household_income']).nlargest(12, 'arbitrage_score').iterrows():
    ax.annotate(row['town'], (row['weighted_flow'], row['median_household_income']),
                fontsize=8.5, xytext=(5, 3), textcoords='offset points')

ax.set_xlabel('Science Center-Weighted Commute Flow (LODES 2021)')
ax.set_ylabel('Median Household Income (ACS 2022)')
ax.set_title('CT Science Center — Commuter Arbitrage\n'
             'Size/color = arbitrage score (flow × income / drive_time^1.5)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))
ax.axvline(plot_df['weighted_flow'].median(), color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(plot_df['median_household_income'].median(), color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(PROCESSED / 'post05_arbitrage_scatter.png', bbox_inches='tight')
plt.show()

## 6. Sensitivity Analysis

A buyer should be able to see whether the ranking holds up under different weightings.
Three scenarios vary α (flow weight) and β (income weight) while holding γ=1.5 constant.

**Robust towns** — appearing in the top 15 across all three scenarios — are the most
defensible recommendations. Towns appearing in only one scenario are flagged as
weighting-sensitive and should not be prioritized without further validation.

In [ ]:
SCENARIOS = {
    'Flow-heavy   (α=1.5, β=0.5)': {'alpha': 1.5, 'beta': 0.5, 'gamma': 1.5},
    'Balanced     (α=1.0, β=1.0)': {'alpha': 1.0, 'beta': 1.0, 'gamma': 1.5},
    'Income-heavy (α=0.5, β=1.5)': {'alpha': 0.5, 'beta': 1.5, 'gamma': 1.5},
}

rankings = {}
for label, w in SCENARIOS.items():
    raw_s = (flow_pct ** w['alpha']) * (income_pct ** w['beta']) / (drive_safe ** w['gamma'])
    scaled = (raw_s - raw_s.min()) / (raw_s.max() - raw_s.min())
    rankings[label] = arb.assign(_s=scaled).nlargest(15, '_s')['town'].tolist()

sensitivity = pd.DataFrame(rankings, index=range(1, 16))
sensitivity.index.name = 'Rank'

# Flag robust towns
all_lists = list(rankings.values())
robust = set(all_lists[0]) & set(all_lists[1]) & set(all_lists[2])
any_one = set(all_lists[0]) | set(all_lists[1]) | set(all_lists[2])
sensitive = any_one - set(all_lists[0]) & set(all_lists[1])  # in at most 1 scenario

print(sensitivity.to_string())
print(f'\n✓ Robust (in all 3): {sorted(robust)}')
print(f'  Count: {len(robust)} towns')

## 7. Drive-Time Bands and Tourism Behaviors

In [ ]:
# Behavior distribution across top 30 arbitrage towns
top30 = arb.nlargest(30, 'arbitrage_score')

band_summary = top30.groupby('tourism_behavior').agg(
    town_count=('town', 'count'),
    avg_drive_min=('drive_time_min', 'mean'),
    avg_weighted_flow=('weighted_flow', 'mean'),
).round(1).sort_values('avg_drive_min')

print('Tourism behavior distribution — top 30 arbitrage towns:')
print(band_summary.to_string())
print()
print('Top 20 towns with drive time + behavior:')
top30[['town', 'drive_time_min', 'drive_band', 'tourism_behavior', 'arbitrage_score']].head(20)

## 8. Multi-Corridor Towns (Appendix)

Towns with significant flow (>500 workers) into multiple anchor cities simultaneously.
These towns have the most "activated" regional travel psychology, but for the Science Center
specifically, multi-corridor exposure to non-Hartford anchors does not increase score.

In [ ]:
THRESHOLD = 500
arb['anchors_n'] = (arb[[f'inbound_to_{a}' for a in ANCHORS]] > THRESHOLD).sum(axis=1)
multi = arb[arb['anchors_n'] >= 2].sort_values('anchors_n', ascending=False)

if not multi.empty:
    mc_plot = multi.nlargest(12, 'total_inbound').set_index('town')
    flow_cols = [f'inbound_to_{a}' for a in ANCHORS]
    mc_plot[flow_cols].rename(columns={
        c: c.replace('inbound_to_','').replace('_',' ').title() for c in flow_cols
    }).plot(kind='barh', stacked=True, figsize=(10, 6), color=colors, alpha=0.85)
    plt.xlabel('Inbound Workers')
    plt.title('Multi-Corridor Towns (≥2 anchors > 500 workers)', fontsize=12)
    plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.legend(title='Anchor', loc='lower right')
    plt.tight_layout()
    plt.savefig(PROCESSED / 'post05_multi_corridor.png', bbox_inches='tight')
    plt.show()
    print(f'{len(multi)} multi-corridor towns; top shown above')
else:
    print('No multi-corridor towns found at this threshold')

## Methodology Note: Archetype Distribution

The top arbitrage towns are predominantly *Affluent Suburban* — this is **expected, not
a discovery**. The arbitrage score weights both income percentile (β) and flow percentile
(α), and high-income towns naturally cluster into the Affluent Suburban archetype.
Reporting this as a finding would be circular.

What the archetype composition tells us: the Science Center's highest-probability audience
is, by construction of this model, a higher-income suburban audience. That's a prior, not
an output. If we had reason to believe the Science Center wanted to reach Working-Class Urban
or Mixed-Income audiences, we'd reduce or remove the income weight (β → 0) and rely on
flow and proximity alone.

## Key Takeaways

**What the data says:**

1. **West Hartford and Glastonbury have the lowest geographic friction to visiting the
   Science Center** — high Hartford commute flow, above-median income, and drive times
   under 15 minutes. The model identifies them as Local Repeat Visitor candidates, meaning
   low per-trip activation cost and strong membership conversion potential.

2. **10 towns are robust across all three weighting scenarios** (Avon, Berlin, Farmington,
   Glastonbury, Rocky Hill, Simsbury, South Windsor, West Hartford, Wethersfield, Windsor).
   These are the most defensible source-market targets regardless of how you weight
   flow vs. income.

3. **Attraction-specific weighting matters.** Under a flat total-inbound ranking, Hamden
   (7,700 New Haven commuters) would outrank West Hartford (6,400 Hartford commuters) for
   a Hartford attraction. With Science Center anchor weights, West Hartford correctly leads.

4. **Drive-time band improvement:** The updated piecewise speed model correctly classifies
   Fairfield and Norwalk as Day-Trippers — the previous single-constant model placed both
   in the Weekender band at inflated times (104 and 116 min vs. actual 72 and 87 min).

**What the data does not say:**

- It does not predict actual visitation. Commute flow predicts geographic familiarity;
  conversion depends on programming, pricing, awareness, and fit.
- It does not control for competing attractions. A town 15 minutes from the Science Center
  is also 15 minutes from many other Hartford-area venues.
- LODES data is 2021 vintage (2-year publication lag); commute patterns may have shifted
  post-pandemic. All claims should cite this vintage.

## Report Output

Saves report-ready files to `outputs/ct_science_center/`. These are the artifacts
the PDF report pipeline pulls from — do not embed data tables in notebook output cells.

In [ ]:
out_df = arb.nlargest(15, 'arbitrage_score')[[
    'town', 'arbitrage_score', 'drive_time_min', 'drive_band',
    'tourism_behavior', 'weighted_flow', 'median_household_income', 'archetype_label'
]].copy().reset_index(drop=True)
out_df.index += 1
out_df.index.name = 'rank'
out_df = out_df.reset_index()

out_df.to_parquet(OUTPUT_DIR / 'top_source_towns.parquet', index=False)
out_df.to_csv(OUTPUT_DIR / 'top_source_towns.csv', index=False)

print(f'Saved -> {OUTPUT_DIR}/top_source_towns.{{parquet,csv}}')
print(f'Rows: {len(out_df)} | Columns: {list(out_df.columns)}')
out_df